In [ ]:
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import utils
import statsmodels.formula.api as smf


audio_paths = [
    f"../music-clips/classical_{i}.wav" for i in range(1,17) ] + [
    f"../music-clips/electronic_{i}.wav" for i in range(1,17)
]
DATA_DIR = Path("../subject-data")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model, processor = utils.load_music_gen_model()
wfs = utils.load_process_bulk_audio(audio_paths, sr=32000)
inputs_dict = utils.process_bulk_music_gen(wfs, processor=processor)

# alter path in dict so it matches piece ids
for old_path in list(inputs_dict.keys()):
    new_path = old_path.replace("../music-clips/", "").replace(".wav", "")
    inputs_dict[new_path] = inputs_dict.pop(old_path)

attentions_dict = utils.extract_attentions(model, inputs_dict)
mad_dict = utils.compute_mad_dict(attentions_dict)
# so now have a dictionary of piece mad for each layer and head


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

In [3]:
# isolate layer 8 and average heads and put in a dict by piece
# for path, mads in mad_dict.items():
    # print(f"{path}: {np.mean(mads[6:8])}")  # average of layer 7 and 8
layer_7_and_8_averages = {path: np.mean(mads[6:8]) for path, mads in mad_dict.items()}
layer_7_averages = {path: np.mean(mads[6]) for path, mads in mad_dict.items()}
layer_8_averages = {path: np.mean(mads[7]) for path, mads in mad_dict.items()}

# print(layer_8_averages.keys())
print(np.std(list(layer_7_and_8_averages.values())))
print(np.std(list(layer_7_averages.values())))
print(np.std(list(layer_8_averages.values())))

0.0021409406655383914
0.002537780587182141
0.0022795741218055604


In [ ]:
pickle_files = sorted(list(DATA_DIR.glob("*.pickle")) + list(DATA_DIR.glob("*.pkl")))
print(f"Found {len(pickle_files)} pickle files")
timeseries_rows = []
trial_rows = []

for pf in pickle_files:
    print("Loading:", pf.name)

    with open(pf, "rb") as f:
        outVars = pickle.load(f)

    expInfo = outVars["expInfo"]
    trialInfo = outVars["trialInfo"]

    subject_id = expInfo["participant_number"]
    n_trials = len(trialInfo["trial"])

    for t in range(n_trials):
        trial_number = trialInfo["trial"][t]
        piece_id = trialInfo["musFile"][t]
        genre = trialInfo["musGenre"][t]
        overall_rating = trialInfo["overall_rating_value"][t]

        dial_vals = trialInfo["dial_values"][t][:3601]
        dial_times = trialInfo["dial_times"][t][:3601]

        if len(dial_vals) != 3601 or len(dial_times) != 3601:
            raise ValueError(f"Trial {trial_number} in {pf.name} does not have 3601 samples.")

        mad_early_seconds = layer_7_averages.get(piece_id, np.nan)
        mad_late_seconds = layer_8_averages.get(piece_id, np.nan)

        # Trial-level row++
        trial_rows.append({
            "subject_id": subject_id,
            "trial_number": trial_number,
            "piece_id": piece_id,
            "genre": genre,
            "overall_rating": overall_rating,
            "mad_early_seconds": mad_early_seconds,
            "mad_late_seconds": mad_late_seconds
        })

        # Time-series rows
        for i in range(3601):
            timeseries_rows.append({
                "subject_id": subject_id,
                "trial_number": trial_number,
                "piece_id": piece_id,
                "genre": genre,
                "time": dial_times[i],
                "sample_index": i,
                "dial_value": dial_vals[i],
                "overall_rating": overall_rating,
                "mad_early_seconds": mad_early_seconds,
                "mad_late_seconds": mad_late_seconds
            })

trial_df = pd.DataFrame(trial_rows)
timeseries_df = pd.DataFrame(timeseries_rows)

print("trial_df shape:", trial_df.shape)
print("timeseries_df shape:", timeseries_df.shape)

/home/kross1/music-integration/music-gen-mad/subject-data
Found 27 pickle files
Loading: s01mi_aud_beh1_out_2026-02-25_16h27.19.437.pickle
Loading: s02mi_aud_beh1_out_2026-02-27_10h09.04.814.pickle
Loading: s03mi_aud_beh1_out_2026-02-27_11h49.36.083.pickle
Loading: s04mi_aud_beh1_out_2026-02-27_13h34.14.663.pickle
Loading: s05mi_aud_beh1_out_2026-03-03_13h43.58.977.pickle
Loading: s07mi_aud_beh1_out_2026-03-05_15h21.07.688.pickle
Loading: s08mi_aud_beh1_out_2026-03-05_16h42.12.368.pickle
Loading: s09mi_aud_beh1_out_2026-03-06_12h17.51.373.pickle
Loading: s10mi_aud_beh1_out_2026-03-06_14h12.07.238.pickle
Loading: s11mi_aud_beh1_out_2026-03-06_15h59.00.555.pickle
Loading: s12mi_aud_beh1_out_2026-03-06_17h38.13.969.pickle
Loading: s13mi_aud_beh1_out_2026-03-06_18h51.55.120.pickle
Loading: s14mi_aud_beh1_out_2026-03-09_09h44.45.235.pickle
Loading: s15mi_aud_beh1_out_2026-03-09_13h52.51.791.pickle
Loading: s16mi_aud_beh1_out_2026-03-10_10h43.24.693.pickle
Loading: s17mi_aud_beh1_out_2026-03

In [ ]:
print("Subjects:", sorted(trial_df["subject_id"].unique()))
print("\nTrials per subject:")
print(trial_df.groupby("subject_id")["trial_number"].nunique())

print("\nMissing MAD values in trial_df:")
print(trial_df[["mad_early_seconds", "mad_late_seconds"]].isna().sum())

trial_out = OUTPUT_DIR / "behavior_trial_level.parquet"
timeseries_out = OUTPUT_DIR / "behavior_timeseries_level.parquet"

trial_df.to_parquet(trial_out, index=False)
timeseries_df.to_parquet(timeseries_out, index=False)

print(f"Saved trial-level data to: {trial_out}")
print(f"Saved timeseries-level data to: {timeseries_out}")

Subjects: ['01', '02', '03', '04', '05', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '24', '25', '27', '28', '29', '31']

Trials per subject:
subject_id
01    32
02    32
03    32
04    32
05    32
07    32
08    32
09    32
10    32
11    32
12    32
13    32
14    32
15    32
16    32
17    32
18    32
19    32
20    32
21    32
22    32
24    32
25    32
27    32
28    32
29    32
31    32
Name: trial_number, dtype: int64

Missing MAD values in trial_df:
mad_early_seconds    0
mad_late_seconds     0
dtype: int64
Saved trial-level data to: ../output/behavior_trial_level.parquet
Saved timeseries-level data to: ../output/behavior_timeseries_level.parquet


In [7]:
# ==========================
# 5) Subject-by-clip summary metrics
# ==========================

def compute_slope(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 2 or np.allclose(x, x[0]):
        return np.nan

    return np.polyfit(x, y, 1)[0]

group_cols = [
    "subject_id", "trial_number", "piece_id", "genre", "overall_rating",
    "mad_early_seconds", "mad_late_seconds"
]

subject_clip_summary = (
    timeseries_df
    .groupby(group_cols, as_index=False)
    .agg(
        mean_continuous_rating=("dial_value", "mean"),
        continuous_sd=("dial_value", lambda x: x.std(ddof=1)),
        continuous_slope=("dial_value", lambda y: compute_slope(
            timeseries_df.loc[y.index, "time"], y
        ))
    )
)

print("subject_clip_summary shape:", subject_clip_summary.shape)
display(subject_clip_summary.head())

subject_clip_summary shape: (864, 10)


,subject_id,trial_number,piece_id,genre,overall_rating,mad_early_seconds,mad_late_seconds,mean_continuous_rating,continuous_sd,continuous_slope
0,01,1,classical_11,classical,0.958621,0.282571,0.293700,0.880241,0.132011,0.006323
1,01,2,classical_12,classical,0.548276,0.282331,0.296777,0.554508,0.043877,0.002223
2,01,3,classical_16,classical,0.682759,0.286818,0.298212,0.660784,0.105568,0.005577
3,01,4,classical_5,classical,0.637931,0.283910,0.296515,0.629012,0.085981,0.000206
4,01,5,classical_15,classical,0.903448,0.281320,0.295139,0.812561,0.102621,0.000920


In [8]:
# ==========================
# 6) Clip-level summary metrics
# ==========================

clip_summary = (
    subject_clip_summary
    .groupby(["piece_id", "genre", "mad_early_seconds", "mad_late_seconds"], as_index=False)
    .agg(
        mean_overall_rating=("overall_rating", "mean"),
        sd_overall_rating=("overall_rating", "std"),
        mean_continuous_rating=("mean_continuous_rating", "mean"),
        sd_continuous_rating=("mean_continuous_rating", "std"),
        mean_continuous_variability=("continuous_sd", "mean"),
        mean_continuous_slope=("continuous_slope", "mean"),
        n_subjects=("subject_id", "nunique")
    )
)

print("clip_summary shape:", clip_summary.shape)
display(clip_summary.head())

clip_summary shape: (32, 11)


,piece_id,genre,mad_early_seconds,mad_late_seconds,mean_overall_rating,sd_overall_rating,mean_continuous_rating,sd_continuous_rating,mean_continuous_variability,mean_continuous_slope,n_subjects
0,classical_1,classical,0.285978,0.298822,0.725764,0.207362,0.696818,0.170989,0.094617,0.002906,27
1,classical_10,classical,0.281808,0.292032,0.624516,0.207599,0.630550,0.139628,0.110269,0.002056,27
2,classical_11,classical,0.282571,0.293700,0.650607,0.216631,0.622951,0.172071,0.105447,0.002923,27
3,classical_12,classical,0.282331,0.296777,0.668718,0.160995,0.646289,0.142685,0.093404,0.003149,27
4,classical_13,classical,0.283588,0.296210,0.654798,0.178246,0.661555,0.128072,0.102317,0.003263,27


In [9]:
# ==========================
# 7) Overall rating disagreement (z-scored within subject)
# ==========================

trial_df_z = trial_df.copy()

trial_df_z["overall_rating_z"] = (
    trial_df_z.groupby("subject_id")["overall_rating"]
    .transform(lambda x: (x - x.mean()) / x.std(ddof=1))
)

overall_disagreement = (
    trial_df_z
    .groupby(["piece_id", "genre", "mad_early_seconds", "mad_late_seconds"], as_index=False)
    .agg(
        overall_disagreement_sd=("overall_rating_z", "std"),
        n_subjects=("subject_id", "nunique")
    )
)

print("overall_disagreement shape:", overall_disagreement.shape)
display(overall_disagreement.head())

overall_disagreement shape: (32, 6)


,piece_id,genre,mad_early_seconds,mad_late_seconds,overall_disagreement_sd,n_subjects
0,classical_1,classical,0.285978,0.298822,0.922368,27
1,classical_10,classical,0.281808,0.292032,0.697041,27
2,classical_11,classical,0.282571,0.293700,0.809554,27
3,classical_12,classical,0.282331,0.296777,0.656225,27
4,classical_13,classical,0.283588,0.296210,0.673446,27


In [10]:
# ==========================
# 8) Continuous disagreement metrics
# ==========================

continuous_disagreement_time = (
    timeseries_df
    .groupby(
        ["piece_id", "genre", "mad_early_seconds", "mad_late_seconds", "sample_index", "time"],
        as_index=False
    )
    .agg(
        variance_across_participants=("dial_value", "var")
    )
)

summary_rows = []

for (piece_id, genre, mad_early, mad_late), g in continuous_disagreement_time.groupby(
    ["piece_id", "genre", "mad_early_seconds", "mad_late_seconds"]
):
    g = g.sort_values("time")

    summary_rows.append({
        "piece_id": piece_id,
        "genre": genre,
        "mad_early_seconds": mad_early,
        "mad_late_seconds": mad_late,
        "mean_continuous_disagreement": g["variance_across_participants"].mean(),
        "time_to_maximal_convergence": g.loc[
            g["variance_across_participants"].idxmin(), "time"
        ]
    })

continuous_disagreement_summary = pd.DataFrame(summary_rows)

print("continuous_disagreement_summary shape:", continuous_disagreement_summary.shape)
display(continuous_disagreement_summary.head())

continuous_disagreement_summary shape: (32, 6)


,piece_id,genre,mad_early_seconds,mad_late_seconds,mean_continuous_disagreement,time_to_maximal_convergence
0,classical_1,classical,0.285978,0.298822,0.009327,9.507750
1,classical_10,classical,0.281808,0.292032,0.025096,30.853875
2,classical_11,classical,0.282571,0.293700,0.060637,56.605937
3,classical_12,classical,0.282331,0.296777,0.010118,8.791806
4,classical_13,classical,0.283588,0.296210,0.101480,4.770086


In [11]:
# ==========================
# 9) Final clip-level analysis table
# ==========================

clip_analysis_df = (
    clip_summary
    .merge(
        overall_disagreement[["piece_id", "overall_disagreement_sd"]],
        on="piece_id",
        how="left"
    )
    .merge(
        continuous_disagreement_summary[[
            "piece_id",
            "mean_continuous_disagreement",
            "time_to_maximal_convergence"
        ]],
        on="piece_id",
        how="left"
    )
)

print("clip_analysis_df shape:", clip_analysis_df.shape)
display(clip_analysis_df.head())

clip_analysis_df shape: (32, 14)


,piece_id,genre,mad_early_seconds,mad_late_seconds,mean_overall_rating,sd_overall_rating,mean_continuous_rating,sd_continuous_rating,mean_continuous_variability,mean_continuous_slope,n_subjects,overall_disagreement_sd,mean_continuous_disagreement,time_to_maximal_convergence
0,classical_1,classical,0.285978,0.298822,0.725764,0.207362,0.696818,0.170989,0.094617,0.002906,27,0.922368,0.009327,9.507750
1,classical_10,classical,0.281808,0.292032,0.624516,0.207599,0.630550,0.139628,0.110269,0.002056,27,0.697041,0.025096,30.853875
2,classical_11,classical,0.282571,0.293700,0.650607,0.216631,0.622951,0.172071,0.105447,0.002923,27,0.809554,0.060637,56.605937
3,classical_12,classical,0.282331,0.296777,0.668718,0.160995,0.646289,0.142685,0.093404,0.003149,27,0.656225,0.010118,8.791806
4,classical_13,classical,0.283588,0.296210,0.654798,0.178246,0.661555,0.128072,0.102317,0.003263,27,0.673446,0.101480,4.770086


In [12]:
# ==========================
# 10) Save outputs
# ==========================

subject_clip_out = OUTPUT_DIR / "behavior_subject_clip_summary.parquet"
clip_summary_out = OUTPUT_DIR / "behavior_clip_summary.parquet"
overall_disagreement_out = OUTPUT_DIR / "overall_disagreement.parquet"
continuous_disagreement_out = OUTPUT_DIR / "continuous_disagreement_summary.parquet"
clip_analysis_out = OUTPUT_DIR / "clip_analysis_ready.parquet"

subject_clip_summary.to_parquet(subject_clip_out, index=False)
clip_summary.to_parquet(clip_summary_out, index=False)
overall_disagreement.to_parquet(overall_disagreement_out, index=False)
continuous_disagreement_summary.to_parquet(continuous_disagreement_out, index=False)
clip_analysis_df.to_parquet(clip_analysis_out, index=False)

print(f"Saved: {subject_clip_out}")
print(f"Saved: {clip_summary_out}")
print(f"Saved: {overall_disagreement_out}")
print(f"Saved: {continuous_disagreement_out}")
print(f"Saved: {clip_analysis_out}")

Saved: ../output/behavior_subject_clip_summary.parquet
Saved: ../output/behavior_clip_summary.parquet
Saved: ../output/overall_disagreement.parquet
Saved: ../output/continuous_disagreement_summary.parquet
Saved: ../output/clip_analysis_ready.parquet


In [15]:
clip_analysis_path = OUTPUT_DIR / "clip_analysis_ready.parquet"
subject_clip_path = OUTPUT_DIR / "behavior_subject_clip_summary.parquet"

clip_df = pd.read_parquet(clip_analysis_path)
subject_clip_df = pd.read_parquet(subject_clip_path)

clip_df["genre"] = pd.Categorical(
    clip_df["genre"],
    categories=["classical", "electronic"]
)

print("clip_df shape:", clip_df.shape)
print("subject_clip_df shape:", subject_clip_df.shape)
display(clip_df.head())

print("Unique clips:", clip_df["piece_id"].nunique())
print("\nGenre counts:")
print(clip_df["genre"].value_counts())

print("\nMissing values:")
print(clip_df.isna().sum())

clip_df shape: (32, 14)
subject_clip_df shape: (864, 10)


,piece_id,genre,mad_early_seconds,mad_late_seconds,mean_overall_rating,sd_overall_rating,mean_continuous_rating,sd_continuous_rating,mean_continuous_variability,mean_continuous_slope,n_subjects,overall_disagreement_sd,mean_continuous_disagreement,time_to_maximal_convergence
0,classical_1,classical,0.285978,0.298822,0.725764,0.207362,0.696818,0.170989,0.094617,0.002906,27,0.922368,0.009327,9.507750
1,classical_10,classical,0.281808,0.292032,0.624516,0.207599,0.630550,0.139628,0.110269,0.002056,27,0.697041,0.025096,30.853875
2,classical_11,classical,0.282571,0.293700,0.650607,0.216631,0.622951,0.172071,0.105447,0.002923,27,0.809554,0.060637,56.605937
3,classical_12,classical,0.282331,0.296777,0.668718,0.160995,0.646289,0.142685,0.093404,0.003149,27,0.656225,0.010118,8.791806
4,classical_13,classical,0.283588,0.296210,0.654798,0.178246,0.661555,0.128072,0.102317,0.003263,27,0.673446,0.101480,4.770086


Unique clips: 32

Genre counts:
genre
classical     16
electronic    16
Name: count, dtype: int64

Missing values:
piece_id                        0
genre                           0
mad_early_seconds               0
mad_late_seconds                0
mean_overall_rating             0
sd_overall_rating               0
mean_continuous_rating          0
sd_continuous_rating            0
mean_continuous_variability     0
mean_continuous_slope           0
n_subjects                      0
overall_disagreement_sd         0
mean_continuous_disagreement    0
time_to_maximal_convergence     0
dtype: int64


In [17]:
# =========================================================
# Participant-level leave-one-out deviation agreement scores
# =========================================================

# Start from participant × clip dataframe
participant_agreement_df = subject_clip_df.copy()

# Add MAD columns from clip_df if they are not already present
mad_cols = ["piece_id", "mad_early_seconds", "mad_late_seconds", "genre"]

cols_to_add = [
    col for col in mad_cols 
    if col not in participant_agreement_df.columns and col in clip_df.columns
]

if cols_to_add:
    participant_agreement_df = participant_agreement_df.merge(
        clip_df[["piece_id"] + cols_to_add],
        on="piece_id",
        how="left"
    )

# Function to compute leave-one-out absolute deviation
def add_leave_one_out_deviation(df, rating_col, new_col):
    df = df.copy()

    group_sum = df.groupby("piece_id")[rating_col].transform("sum")
    group_n = df.groupby("piece_id")[rating_col].transform("count")

    loo_mean = (group_sum - df[rating_col]) / (group_n - 1)

    df[new_col] = (df[rating_col] - loo_mean).abs()

    return df

# Overall rating agreement
participant_agreement_df = add_leave_one_out_deviation(
    participant_agreement_df,
    rating_col="overall_rating",
    new_col="overall_rating_loo_deviation"
)

# Mean continuous rating agreement, if available
if "mean_continuous_rating" in participant_agreement_df.columns:
    participant_agreement_df = add_leave_one_out_deviation(
        participant_agreement_df,
        rating_col="mean_continuous_rating",
        new_col="mean_continuous_rating_loo_deviation"
    )

# Make categorical variables ready for mixed models
participant_agreement_df["subject_id"] = participant_agreement_df["subject_id"].astype("category")
participant_agreement_df["piece_id"] = participant_agreement_df["piece_id"].astype("category")
participant_agreement_df["genre"] = participant_agreement_df["genre"].astype("category")

# Check
display(participant_agreement_df.head())
print(participant_agreement_df.columns.tolist())

# Save
participant_agreement_path = OUTPUT_DIR / "participant_leave_one_out_agreement_scores.parquet"
participant_agreement_df.to_parquet(participant_agreement_path, index=False)

print("Saved to:", participant_agreement_path.resolve())

,subject_id,trial_number,piece_id,genre,overall_rating,mad_early_seconds,mad_late_seconds,mean_continuous_rating,continuous_sd,continuous_slope,overall_rating_loo_deviation,mean_continuous_rating_loo_deviation
0,01,1,classical_11,classical,0.958621,0.282571,0.293700,0.880241,0.132011,0.006323,0.319860,0.267186
1,01,2,classical_12,classical,0.548276,0.282331,0.296777,0.554508,0.043877,0.002223,0.125074,0.095312
2,01,3,classical_16,classical,0.682759,0.286818,0.298212,0.660784,0.105568,0.005577,0.029214,0.016037
3,01,4,classical_5,classical,0.637931,0.283910,0.296515,0.629012,0.085981,0.000206,0.013539,0.013293
4,01,5,classical_15,classical,0.903448,0.281320,0.295139,0.812561,0.102621,0.000920,0.172807,0.106974


['subject_id', 'trial_number', 'piece_id', 'genre', 'overall_rating', 'mad_early_seconds', 'mad_late_seconds', 'mean_continuous_rating', 'continuous_sd', 'continuous_slope', 'overall_rating_loo_deviation', 'mean_continuous_rating_loo_deviation']
Saved to: /home/kross1/music-integration/music-gen-mad/output/participant_leave_one_out_agreement_scores.parquet


In [20]:
#Mixed Linear Model Regression Results
agreement_model_df = participant_agreement_df[
    [
        "subject_id",
        "genre",
        "overall_rating_loo_deviation",
        "mad_early_seconds",
        "mad_late_seconds"
    ]
].dropna().copy()

agreement_model = smf.mixedlm(
    "overall_rating_loo_deviation ~ mad_early_seconds + mad_late_seconds + C(genre)",
    data=agreement_model_df,
    groups=agreement_model_df["subject_id"]
)

agreement_result = agreement_model.fit(reml=False)
print(agreement_result.summary())

                  Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: overall_rating_loo_deviation
No. Observations: 864     Method:             ML                          
No. Groups:       27      Scale:              0.0130                      
Min. group size:  32      Log-Likelihood:     629.6355                    
Max. group size:  32      Converged:          Yes                         
Mean group size:  32.0                                                    
---------------------------------------------------------------------------
                           Coef.   Std.Err.    z     P>|z|   [0.025  0.975]
---------------------------------------------------------------------------
Intercept                   0.687     0.599   1.147  0.251   -0.487   1.861
C(genre)[T.electronic]     -0.018     0.010  -1.677  0.094   -0.038   0.003
mad_early_seconds           5.434     2.479   2.192  0.028    0.576  10.292
mad_late_seconds           -6.964     

/tmp/ipykernel_3615747/125080462.py:18: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  agreement_result = agreement_model.fit(reml=False)


In [21]:
[name for name in globals().keys() if "model" in name.lower() or "result" in name.lower() or "fit" in name.lower()]

['model', 'agreement_model_df', 'agreement_model', 'agreement_result']

In [24]:
# =========================================================
# Participant-level mixed-effects model: overall rating
# overall_rating ~ early MAD + late MAD + genre + (1 | subject)
# asks "Do early MAD, late MAD, and genre predict participants’ overall ratings, while accounting for the fact that each participant gives multiple ratings?"
# =========================================================

df_long = pd.read_parquet(f"{OUTPUT_DIR}/behavior_subject_clip_summary.parquet")

df_long["genre"] = df_long["genre"].astype("category")
df_long["subject_id"] = df_long["subject_id"].astype("category")
df_long["piece_id"] = df_long["piece_id"].astype("category")

df_long.head()

overall_mixed_df = df_long[
    [
        "subject_id",
        "piece_id",
        "genre",
        "overall_rating",
        "mad_early_seconds",
        "mad_late_seconds"
    ]
].dropna().copy()

overall_mixed_df["genre"] = overall_mixed_df["genre"].astype("category")
overall_mixed_df["subject_id"] = overall_mixed_df["subject_id"].astype("category")

model = smf.mixedlm(
    "overall_rating ~ mad_early_seconds + mad_late_seconds + C(genre)",
    data=overall_mixed_df,
    groups=overall_mixed_df["subject_id"]
)

result = model.fit(reml=False)
print(result.summary())

               Mixed Linear Model Regression Results
Model:                MixedLM   Dependent Variable:   overall_rating
No. Observations:     864       Method:               ML            
No. Groups:           27        Scale:                0.0391        
Min. group size:      32        Log-Likelihood:       147.4492      
Max. group size:      32        Converged:            Yes           
Mean group size:      32.0                                          
--------------------------------------------------------------------
                        Coef.  Std.Err.   z    P>|z|  [0.025  0.975]
--------------------------------------------------------------------
Intercept                0.214    1.040  0.206 0.837  -1.824   2.252
C(genre)[T.electronic]  -0.000    0.018 -0.008 0.993  -0.036   0.035
mad_early_seconds      -27.175    4.302 -6.317 0.000 -35.607 -18.744
mad_late_seconds        27.457    3.776  7.272 0.000  20.057  34.857
Group Var                0.008    0.013           

/tmp/ipykernel_3615747/2293326186.py:35: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  result = model.fit(reml=False)
